# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [ ]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# import os
# os.environ["PATH"] += os.pathsep + "/home/igadiya/.local/bin"

# # Create a virtual environment
# !uv venv .venv --seed --clear

# # Install dependencies (aligned with starter notebook)
# !.venv/bin/python -m pip install \
#      "numpy<2" \
#      "torch==2.2.2" \
#      "transformers>=4.51,<4.57" \
#      "tokenizers>=0.21,<0.22" \
#      "accelerate==0.30.1" \
#      sympy tqdm antlr4-python3-runtime==4.11.1 ipykernel jupyter sentencepiece \
#      "bitsandbytes" \
#      "pandas"\
#      "peft"

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding.")
# print("Selection process: on top right, click on current kernel '(usually named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

### Run the cell below every time to activate the installed environment. 

In [1]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [2]:
import json
import os
import re
import sys
from pathlib import Path
from typing import Optional

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
#MAX_TOKENS  = 4096
MAX_TOKENS_REFLECT  = 1000
MAX_TOKENS_THINK = 500
#MAX_TOKENS  = 500

DEVICE = "cuda"

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [3]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [4]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician reasoning.\n\n "
    "MANDATORY RULE:\n"
    "- The FIRST token MUST be: \\boxed{final_answer}\n "
    "- If multiple sub-answers are required, put them in one box separated by commas, e.g. \\boxed{3, 7}.\n "
    "- If this is not the first token, the response is invalid.\n\n"
    "FORMAT:\n"
    "\\boxed{final_answer}\n"
    "Brief justification (<= 2 sentences).\n\n "
    "CONSTRAINTS:\n"
    "- Do NOT think before answering.\n"
    "- Do NOT delay the boxed answer.\n"
    "- Output EXACTLY 2 lines.\n"
    "- No extra text.\n"
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician reasoning.\n\n"
    "MANDATORY RULE:\n"
    "- The FIRST token MUST be: \\boxed{X} where X is the answer choice.\n "
    "- If this is not the first token, the response is invalid.\n\n"
    "FORMAT:\n"
    "\\boxed{X}\n"
    "Brief justification (<= 2 sentences).\n\n"
    "CONSTRAINTS:\n"
    "- Do NOT think before answering.\n"
    "- Do NOT delay the boxed answer.\n"
    "- Output EXACTLY 2 lines.\n"
    "- No extra text.\n"
)

SYSTEM_PROMPT_REFLECTION_FREE = (
    "Pick ONLY ONE method to reevaluate the answer to the question."
    "MANDATORY RULE:"
    "- Final answer MUST be: \\boxed{X} where X = correct answer."
    "- If multiple sub-answers are required, put them in one box separated by commas, e.g. \\boxed{3, 7}."
    "- \\boxed{answer} MUST EXIST within the first 100 tokens"
    "CONSTRAINTS:"
    "- DO NOT evaluate using more than ONE METHOD"
)

SYSTEM_PROMPT_REFLECTION_MCQ = (
    "Pick ONLY ONE method to reevaluate the answer to the question."
    "MANDATORY RULE:\n"
    "- Final answer MUST be: \\boxed{X} where X = correct option."
    "- \\boxed{answer} MUST EXIST within the first 100 tokens"
    "CONSTRAINTS:\n"
    "- DO NOT evaluate using more than ONE METHOD"
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question

def build_reflection_prompt(question: str, options: Optional[list]) -> tuple[str, str, str]:
    """Return (system_prompt, user_prompt, reflection_prompt) for a question."""
    system_prompt, user_prompt = build_prompt(question, options)
    if options:
        return system_prompt, user_prompt, SYSTEM_PROMPT_REFLECTION_MCQ
    return system_prompt, user_prompt, SYSTEM_PROMPT_REFLECTION_FREE

# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $2a$
H. $frac{3}{2a}$
I. $frac{3}{2a^2}$
J. ...

── Free-form user prompt (first 200 chars) ──
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS] ...



## 5. Load Model with vLLM

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [5]:
import gc

# Delete model and tokenizer if they exist
try:
    del model
    del tokenizer
except:
    pass

# Clear Python garbage
gc.collect()

# Clear PyTorch CUDA cache
torch.cuda.empty_cache()

# Reset cached memory stats
torch.cuda.reset_peak_memory_stats()

print("Memory cleared!")
print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Cached: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

Memory cleared!
Allocated: 0.00 GB
Cached: 0.00 GB


In [6]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    use_fast=False,
    padding_side='left',
)

In [7]:
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(load_in_8bit=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

def generate_batch(prompts: list[str], max_new_tokens: int) -> str:
    #inputs = tokenizer(prompt_text, return_tensors="pt").to(DEVICE)
    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(DEVICE)

    #changed temp to 0.3
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.3,
            min_p=0.1,
            #top_p=0.95,
            #top_k=20,
            repetition_penalty=1.0,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        
    responses = []
    i = 0
    for output in tqdm(outputs, desc="Decoding"):
        print("Q-num:", i)
        generated_ids = output[inputs["input_ids"].shape[1]:]
        responses.append(tokenizer.decode(generated_ids, skip_special_tokens=True).strip())
        i+=1

    return responses

print(f"Model loaded on {DEVICE}.")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Model loaded on cuda.


## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

In [ ]:
import pandas as pd
temp_output_file = "data/responses_checkpoint.csv"
batch_size = 2

# 1. Load existing progress if it exists
if os.path.exists(temp_output_file):
    df_existing = pd.read_csv(temp_output_file)
    completed_ids = set(df_existing['id'].tolist())
    print(f"Resuming from checkpoint. {len(completed_ids)} items already processed.")
else:
    completed_ids = set()
    pd.DataFrame(columns=["id", "question", "initial_response", "revised_response" ,"has_boxed"]).to_csv(temp_output_file, index=False)

remaining_data = [item for item in data if item.get("id") not in completed_ids]

for i in range(0, len(remaining_data), batch_size):
    items = remaining_data[i : i + batch_size]

    prompts = []
    for item in items:
        system, user = build_prompt(item["question"], item.get("options"))
        prompt_text = tokenizer.apply_chat_template(
            [{"role": "system", "content": system},
             {"role": "user",   "content": user}],
            tokenize=False,
            add_generation_prompt=True,
        )
        prompts.append(prompt_text)

    all_responses = []
    batch_out_all = generate_batch(prompts, MAX_TOKENS_THINK)
    all_responses.extend(batch_out_all)

    revised_prompts = []
    for j, item in enumerate(items):
        system, user, reflection = build_reflection_prompt(item["question"], item.get("options"))
        prompt_text = tokenizer.apply_chat_template(
            [{"role": "system", "content": system},
             {"role": "user",   "content": user},
             {"role": "assistant", "content": all_responses[j]},
             {"role": "user",   "content": reflection}],
            tokenize=False,
            add_generation_prompt=True,
        )
        revised_prompts.append(prompt_text)

    all_revised_responses = []
    batch_out_revised = generate_batch(revised_prompts, MAX_TOKENS_REFLECT)
    all_revised_responses.extend(batch_out_revised)

    results = []
    for j, item in enumerate(items):
        results.append({
            "id": item.get("id"),
            "question": item.get("question"),
            "initial_response": all_responses[j],
            "revised_response": all_revised_responses[j],
            "has_box": '\\boxed{' in all_revised_responses[j]
        })
    
    df_batch = pd.DataFrame(results)
    df_batch.to_csv(temp_output_file, mode='a', index=False, header=False)

    for k in range(len(all_revised_responses)):
        print(f"\n── Response {k} (id={data[k].get('id')}) ──")

Resuming from checkpoint. 2 items already processed.


Decoding: 100%|██████████| 2/2 [00:00<00:00, 171.12it/s]


Q-num: 0
Q-num: 1


## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [ ]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data, all_revised_responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

## 8. Summary

Print accuracy broken down by question type.

In [ ]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [ ]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!